# Limpieza y transformación del dataset

## Objetivo

Realizar las transformaciones necesarias para mejorar la calidad del conjunto de datos, corregir inconsistencias identificadas durante la exploración y generar un dataset limpio que sirva como base para el análisis y la construcción del dashboard.

In [20]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)

In [21]:
ruta = Path("../data/raw/3._Accidentes_de_Transito_ocurridos_en_el_Municipio_de_Bucaramanga_20260704.csv")

df = pd.read_csv(ruta, encoding="utf-8")

df.head()

,ORDEN,FECHA,AÑO,MES,DÍA,GRAVEDAD,PEATON,AUTOMOVIL,CAMPERO,CAMIONETA,MICRO,BUSETA,BUS,CAMION,VOLQUETA,MOTO,BICICLETA,OTRO,BARRIO,HORA,ENTIDAD,COMUNA,Propietario de Vehículo,DIURNIO/NOCTURNO
0,1,2012-01-01T00:00:00.000,2012,01. Enero,07. Domingo,Con heridos,0,1,0,0,0,0,0,0,0,0,0,0,Mutis,1899-12-31T12:15:00.000,AGENTES DTB,17. MUTIS,Particular,Diurno
1,2,2012-01-01T00:00:00.000,2012,01. Enero,07. Domingo,Solo daños,0,1,0,1,0,0,0,0,0,0,0,0,Regaderos Norte,1899-12-31T14:00:00.000,AGENTES DTB,02. NORORIENTAL,Empresa,Diurno
2,3,2012-01-01T00:00:00.000,2012,01. Enero,07. Domingo,Solo daños,0,0,0,1,0,0,0,0,0,0,0,0,Cabecera Del Llano,1899-12-31T12:00:00.000,AGENTES DTB,12. CABECERA DEL LLANO,Particular,Diurno
3,4,2012-01-01T00:00:00.000,2012,01. Enero,07. Domingo,Solo daños,0,1,0,1,0,0,0,0,0,0,0,0,Norte Bajo,1899-12-31T18:30:00.000,AGENTES DTB,03. SAN FRANCISCO,Empresa,Nocturno
4,5,2012-01-01T00:00:00.000,2012,01. Enero,07. Domingo,Con heridos,1,0,0,0,0,0,0,0,0,1,0,0,Dangond,1899-12-31T00:30:00.000,AGENTES DTB,11. SUR,Particular,Nocturno


In [22]:
df["FECHA"].head()

0    2012-01-01T00:00:00.000
1    2012-01-01T00:00:00.000
2    2012-01-01T00:00:00.000
3    2012-01-01T00:00:00.000
4    2012-01-01T00:00:00.000
Name: FECHA, dtype: str

In [23]:
df["FECHA"] = pd.to_datetime(df["FECHA"])

In [24]:
df["FECHA"].dtype

dtype('<M8[us]')

In [25]:
df["FECHA"].head()

0   2012-01-01
1   2012-01-01
2   2012-01-01
3   2012-01-01
4   2012-01-01
Name: FECHA, dtype: datetime64[us]

### Conversión de la variable `FECHA`

La columna `FECHA` se encontraba almacenada como texto (`str`), lo que limitaba la realización de análisis temporales.

Se convirtió al tipo de dato `datetime`, permitiendo utilizar las funciones de fechas de Pandas y facilitando análisis por año, mes, día y otros componentes temporales.

In [26]:
df["HORA"].head(10)

0    1899-12-31T12:15:00.000
1    1899-12-31T14:00:00.000
2    1899-12-31T12:00:00.000
3    1899-12-31T18:30:00.000
4    1899-12-31T00:30:00.000
5    1899-12-31T21:10:00.000
6    1899-12-31T13:50:00.000
7    1899-12-31T19:20:00.000
8    1899-12-31T12:00:00.000
9    1899-12-31T16:35:00.000
Name: HORA, dtype: str

In [27]:
df["HORA"].str[:10].value_counts()

HORA
1899-12-31    39190
2                 1
6.3               1
7.4               1
Name: count, dtype: int64

### Observación

Durante la validación de la variable `HORA` se identificó que **39.190 registros** siguen el formato esperado (`1899-12-31THH:MM:SS.000`), mientras que **3 registros presentan un formato inconsistente**, por lo que deberán ser revisados antes de realizar la conversión definitiva de la variable.

In [28]:
df.loc[~df["HORA"].str.startswith("1899-12-31"), ["ORDEN", "FECHA", "HORA"]]

,ORDEN,FECHA,HORA
39113,39114,2022-07-01,2
39156,39157,2023-03-10,6.3
39169,39170,2023-05-18,7.4


In [29]:
df["HORA"] = pd.to_datetime(df["HORA"], errors="coerce")

In [30]:
df["HORA"].isna().sum()

np.int64(3)

In [31]:
df["HORA"] = df["HORA"].dt.time

### Transformación de la variable `HORA`

Se identificó que la mayoría de los registros almacenaban la hora junto con una fecha fija (`1899-12-31`), utilizada únicamente como referencia por el sistema de origen.

Durante la conversión al tipo temporal se detectaron **3 registros con un formato inválido** (`2`, `6.3` y `7.4`). Debido a que no fue posible inferir su valor real sin introducir supuestos, estos registros fueron convertidos a valores temporales nulos (`NaT`).

Posteriormente, se conservó únicamente el componente correspondiente a la hora para el resto de los registros.

In [34]:
df["GRAVEDAD"].value_counts()

GRAVEDAD
Solo daños     19602
Con heridos    18982
Con muertos      503
Con Muertos      106
Name: count, dtype: int64

In [35]:
df["GRAVEDAD"] = df["GRAVEDAD"].replace({
    "Con Muertos": "Con muertos"
})

In [36]:
df["GRAVEDAD"].value_counts()

GRAVEDAD
Solo daños     19602
Con heridos    18982
Con muertos      609
Name: count, dtype: int64

### Estandarización de la variable `GRAVEDAD`

Durante la exploración se identificó una inconsistencia en la escritura de la categoría **"Con muertos"**, la cual también aparecía registrada como **"Con Muertos"**.

Se unificaron ambas etiquetas en una única categoría con el fin de evitar duplicidades durante el análisis y la visualización de los datos.

In [37]:
df.columns

Index(['ORDEN', 'FECHA', 'AÑO', 'MES', 'DÍA', 'GRAVEDAD', 'PEATON',
       'AUTOMOVIL', 'CAMPERO', 'CAMIONETA', 'MICRO', 'BUSETA', 'BUS', 'CAMION',
       'VOLQUETA', 'MOTO', 'BICICLETA', 'OTRO', 'BARRIO', 'HORA', 'ENTIDAD',
       'COMUNA', 'Propietario de Vehículo', 'DIURNIO/NOCTURNO'],
      dtype='str')

In [38]:
df.rename(columns={
    "DIURNIO/NOCTURNO": "DIURNO/NOCTURNO"
}, inplace=True)

In [40]:
MESES = {
    1: "Enero",
    2: "Febrero",
    3: "Marzo",
    4: "Abril",
    5: "Mayo",
    6: "Junio",
    7: "Julio",
    8: "Agosto",
    9: "Septiembre",
    10: "Octubre",
    11: "Noviembre",
    12: "Diciembre"
}

df["MES"] = df["FECHA"].dt.month
df["MES_NOMBRE"] = df["MES"].map(MESES)

In [41]:
DIAS = {
    0: "Lunes",
    1: "Martes",
    2: "Miércoles",
    3: "Jueves",
    4: "Viernes",
    5: "Sábado",
    6: "Domingo"
}

df["DIA_SEMANA"] = df["FECHA"].dt.dayofweek
df["DIA_NOMBRE"] = df["DIA_SEMANA"].map(DIAS)

In [42]:
df["FIN_DE_SEMANA"] = df["DIA_SEMANA"].isin([5, 6])

In [43]:
df.columns

Index(['ORDEN', 'FECHA', 'AÑO', 'MES', 'DÍA', 'GRAVEDAD', 'PEATON',
       'AUTOMOVIL', 'CAMPERO', 'CAMIONETA', 'MICRO', 'BUSETA', 'BUS', 'CAMION',
       'VOLQUETA', 'MOTO', 'BICICLETA', 'OTRO', 'BARRIO', 'HORA', 'ENTIDAD',
       'COMUNA', 'Propietario de Vehículo', 'DIURNO/NOCTURNO', 'MES_NOMBRE',
       'DIA_SEMANA', 'DIA_NOMBRE', 'FIN_DE_SEMANA'],
      dtype='str')

In [44]:
df.drop(columns=["DÍA"], inplace=True)

In [45]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 39193 entries, 0 to 39192
Data columns (total 27 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   ORDEN                    39193 non-null  int64         
 1   FECHA                    39193 non-null  datetime64[us]
 2   AÑO                      39193 non-null  int64         
 3   MES                      39193 non-null  int32         
 4   GRAVEDAD                 39193 non-null  str           
 5   PEATON                   39193 non-null  int64         
 6   AUTOMOVIL                39193 non-null  int64         
 7   CAMPERO                  39193 non-null  int64         
 8   CAMIONETA                39193 non-null  int64         
 9   MICRO                    39193 non-null  int64         
 10  BUSETA                   39193 non-null  int64         
 11  BUS                      39193 non-null  int64         
 12  CAMION                   39193 non-null  in

In [47]:
df["AÑO"] = df["FECHA"].dt.year

In [48]:
df[["MES", "MES_NOMBRE"]].drop_duplicates().sort_values("MES")

,MES,MES_NOMBRE
0,1,Enero
317,2,Febrero
693,3,Marzo
1087,4,Abril
1429,5,Mayo
1781,6,Junio
2102,7,Julio
2436,8,Agosto
2820,9,Septiembre
3203,10,Octubre


In [49]:
df[["DIA_SEMANA", "DIA_NOMBRE"]].drop_duplicates().sort_values("DIA_SEMANA")

,DIA_SEMANA,DIA_NOMBRE
6,0,Lunes
13,1,Martes
19,2,Miércoles
29,3,Jueves
41,4,Viernes
50,5,Sábado
0,6,Domingo


In [50]:
df["FIN_DE_SEMANA"].value_counts()

FIN_DE_SEMANA
False    28596
True     10597
Name: count, dtype: int64

In [52]:
from pathlib import Path

ruta_salida = Path("../data/processed/accidentes_limpio.csv")

df.to_csv(ruta_salida, index=False, encoding="utf-8")

# Conclusiones

Durante la etapa de limpieza y transformación se preparó el conjunto de datos para su posterior análisis, obteniendo los siguientes resultados:

- Se convirtieron las variables `FECHA` y `HORA` a formatos temporales adecuados, facilitando el análisis de la información en función del tiempo.
- Se identificaron y documentaron tres registros con valores de hora inválidos, los cuales fueron convertidos a valores nulos (`NaT`) para preservar la integridad de los datos sin realizar suposiciones.
- Se estandarizó la variable `GRAVEDAD`, eliminando inconsistencias en la escritura de sus categorías.
- Se corrigió el nombre de la columna `DIURNO/NOCTURNO` y se generaron nuevas variables derivadas (`MES`, `MES_NOMBRE`, `DIA_SEMANA`, `DIA_NOMBRE` y `FIN_DE_SEMANA`) para facilitar el análisis temporal.
- Como resultado, se obtuvo un conjunto de datos consistente, estructurado y listo para la etapa de análisis exploratorio y la construcción del dashboard en Power BI.